# Notebook 01: Data Setup & Pipeline Validation
## Phase 1 of the Experimental Workflow (`results/final_experimental_report.md`)

**Objectives:**
1. **T1.1**: Verify the CIC-IoMT-2024 dataset (13 target classes) and label mapping.
2. **T1.2**: Extract the $T=10$ packet spatio-temporal window for the 6 header fields (`PL, IAT, DIR, WIN, TTL, FLG`).
3. **T1.3**: Verify the deterministic stratified split at `seed=0` (70% Train+Validation / 30% Test; Validation is 10% of Train, i.e., 63% / 7% / 30%).
4. **T1.4**: Instantiate the PyTorch `DataLoader` objects ($B=64$) and inspect `0.0` padding vs. physical values scaled to $[0.001, 1.0]$.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import torch
import pandas as pd

# Resolve project paths
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils import seed_everything
seed_everything(seed=0)
print(f"Repo Root: {REPO_ROOT}")
print("Global determinism configured with seed=0.")

### 1.1 Dataset Discovery & Metadata Check (T1.1)
Search for the CIC-IoMT-2024 dataset in standard cluster NAS (`/nas/data/datasets/cic-iomt/`) or local `data/` paths.

In [ ]:
DATA_CANDIDATES = [
    Path('/nas/data/datasets/cic-iomt'),
    REPO_ROOT / 'dataset',
    REPO_ROOT / 'data',
    Path.home() / 'data' / 'cic-iomt'
]

DATA_DIR = None
for p in DATA_CANDIDATES:
    if p.exists():
        DATA_DIR = p
        break

if DATA_DIR is None:
    DATA_DIR = REPO_ROOT / 'data'
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print(f"[WARN] Dataset directory not found in standard paths. Defaulting to {DATA_DIR}")
else:
    print(f"[OK] Dataset located at: {DATA_DIR}")

# Check classes.txt and label_conv.json
CLASSES_FILE = DATA_DIR / 'classes.txt'
if CLASSES_FILE.exists():
    with open(CLASSES_FILE) as f:
        classes = [line.strip() for line in f if line.strip()]
    print(f"[OK] Found {len(classes)} classes in classes.txt:")
    for idx, c in enumerate(classes):
        print(f"  {idx:2d}: {c}")
else:
    print("[INFO] classes.txt will be generated on first dataset load.")

### 1.2 Input Configuration: $T=10$ Packets, $F=6$ Header Fields (T1.2)
Configure the networking dataset parameters according to the frozen protocol in `results/final_experimental_report.md`.

In [ ]:
NUM_PKTS = 10
FIELDS = ['PL', 'IAT', 'DIR', 'WIN', 'TTL', 'FLG']
NUM_CLASSES = 13
BATCH_SIZE = 64
SEED = 0

print("Spatio-temporal input configuration:")
print(f"- Time horizon T: {NUM_PKTS} packets")
print(f"- Header fields F ({len(FIELDS)}): {FIELDS}")
print(f"- Task: Multiclass ({NUM_CLASSES} classes)")
print(f"- Seed: {SEED}")

### 1.3 DataLoader Creation & Inspection (T1.3 & T1.4)
Verify that loaders produce tensors of shape `[batch_size, 1, 10, 6]` with semantic zero padding (`0.0`).

In [ ]:
from datasets.data_loader import get_loaders

trn_loader, val_loader, tst_loader, taskcla = get_loaders(
    datasets=['cic-iomt'],
    num_tasks=1,
    nc_first_task=NUM_CLASSES,
    nc_incr_tasks=0,
    batch_size=BATCH_SIZE,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    seed=SEED,
    num_pkts=NUM_PKTS,
    fields=FIELDS
)
print("[SUCCESS] DataLoaders instantiated successfully!")
print(f"  Train batches: {len(trn_loader[0])}")
print(f"  Val batches:   {len(val_loader[0])}")
print(f"  Test batches:  {len(tst_loader[0])}")

# Inspect the first batch
images, targets = next(iter(trn_loader[0]))
print(f"\nInput batch shape:   {images.shape} (Expected: [{BATCH_SIZE}, 1, {NUM_PKTS}, {len(FIELDS)}])")
print(f"Target batch shape:  {targets.shape}")
print(f"Input min/max range: [{images.min():.4f}, {images.max():.4f}]")
print(f"Zero-padding ratio:  {(images == 0.0).float().mean() * 100:.2f}%")